# News-Sentiment Fine-Tuning — Setup & Inspection

**Goal of this notebook (Iteration 0–1):** confirm we can load the dataset and a
Hugging Face model, and *inspect both* before training anything.

Plan reference: `docs/news_sentiment_finetuning_plan.md`.

Order of operations:
1. Imports + environment check
2. Load `financial_phrasebank` and inspect it (schema, labels, samples, class balance)
3. Load a tokenizer + classification model and inspect them
4. Tokenize a sample and run one forward pass (untrained) to prove the wiring works

In [1]:
# If running on a fresh machine / Colab, uncomment:
# %pip install -q transformers datasets evaluate accelerate scikit-learn

import platform

import numpy as np
import pandas as pd
import torch
import transformers
import datasets

print("Python      :", platform.python_version())
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets    :", datasets.__version__)

# Pick the best available device (CUDA > Apple MPS > CPU).
if torch.cuda.is_available():
    DEVICE = "cuda"
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print("device      :", DEVICE)

/Users/armandoordoricadelatorre/miniconda/envs/sentiment-ltr-paper/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python      : 3.11.15
torch       : 2.12.1
transformers: 5.12.1
datasets    : 5.0.0
device      : mps


## 1. Load the dataset — Financial PhraseBank

~4.8k finance sentences labeled **negative / neutral / positive**.

> **Compatibility gotcha (good interview talking point):** the canonical
> `financial_phrasebank` repo ships as a *Python loading script*, and `datasets`
> v4/v5 **removed script support** (`RuntimeError: Dataset scripts are no longer
> supported`). `trust_remote_code=True` was also removed. The fix is to load a
> **script-free Parquet mirror**. We use `atrost/financial_phrasebank`, which has
> proper `ClassLabel` names and ready-made train/validation/test splits
> (3100 / 776 / 970 = 4846, matching the original `sentences_50agree` config).

In [2]:
from datasets import load_dataset, ClassLabel

# Script-free Parquet mirrors (the canonical script-based repo no longer loads on
# datasets v5 — see the note above).
PRIMARY_SOURCE = "atrost/financial_phrasebank"          # ClassLabel + train/val/test splits
FALLBACK_SOURCE = "warwickai/financial_phrasebank_mirror"  # single train split, int labels
LABEL_NAMES_FALLBACK = ["negative", "neutral", "positive"]


def load_phrasebank():
    """Load a script-free Financial PhraseBank (datasets v5 compatible)."""
    try:
        return load_dataset(PRIMARY_SOURCE)
    except Exception as exc:
        print(f"Primary source failed ({type(exc).__name__}); falling back to mirror.")
        ds = load_dataset(FALLBACK_SOURCE)
        # Normalize plain int labels to a ClassLabel so id<->name mapping works downstream.
        if not isinstance(ds["train"].features["label"], ClassLabel):
            ds = ds.cast_column("label", ClassLabel(names=LABEL_NAMES_FALLBACK))
        return ds


raw = load_phrasebank()
raw

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label'],
        num_rows: 3100
    })
    validation: Dataset({
        features: ['sentence', 'label'],
        num_rows: 776
    })
    test: Dataset({
        features: ['sentence', 'label'],
        num_rows: 970
    })
})

### 1a. Inspect the dataset

Always look at the data before touching the model: schema, label meanings, a few raw
rows, and the class balance.

In [3]:
train = raw["train"]

# Schema + label definitions (ClassLabel maps int id -> human name).
label_feature = train.features["label"]
LABEL_NAMES = label_feature.names
ID2LABEL = {i: name for i, name in enumerate(LABEL_NAMES)}
LABEL2ID = {name: i for i, name in ID2LABEL.items()}

print("Features   :", train.features)
print("Num rows   :", train.num_rows)
print("Label names:", LABEL_NAMES)
print("id2label   :", ID2LABEL)

Features   : {'sentence': Value('string'), 'label': ClassLabel(names=['negative', 'neutral', 'positive'])}
Num rows   : 3100
Label names: ['negative', 'neutral', 'positive']
id2label   : {0: 'negative', 1: 'neutral', 2: 'positive'}


In [4]:
# A few sample rows with decoded labels.
sample = train.shuffle(seed=42).select(range(8))
sample_df = pd.DataFrame({
    "sentence": sample["sentence"],
    "label_id": sample["label"],
    "label": [ID2LABEL[i] for i in sample["label"]],
})
sample_df

,sentence,label_id,label
0,cents Scout for potential acquisition targets ...,1,neutral
1,"However , the brokers ' ratings on the stock d...",1,neutral
2,Investment management and investment advisory ...,1,neutral
3,As a result of these negotiations the company ...,0,negative
4,The company reported net sales of 302 mln euro...,1,neutral
5,The total investment in the company will be EU...,1,neutral
6,Pretax profit rose to EUR 0.6 mn from EUR 0.4 ...,2,positive
7,The total headcount reduction will be 50 perso...,0,negative


In [5]:
# Class balance — Financial PhraseBank is imbalanced (neutral dominates).
# This is why we'll track macro-F1, not just accuracy, when training.
counts = pd.Series(train["label"]).map(ID2LABEL).value_counts()
balance = pd.DataFrame({
    "count": counts,
    "pct": (counts / counts.sum() * 100).round(1),
})
print(balance)

# Token-length sense check (rough, by whitespace) to pick a sane max_length later.
word_lens = pd.Series([len(s.split()) for s in train["sentence"]])
print("\nWords per sentence — describe():")
print(word_lens.describe()[["mean", "50%", "max"]].round(1))

          count   pct
neutral    1852  59.7
positive    866  27.9
negative    382  12.3

Words per sentence — describe():
mean    23.1
50%     21.0
max     81.0
dtype: float64


## 2. Load the model + tokenizer

Start with `distilbert-base-uncased` (small, fast baseline). We pass the label maps so
the model config carries human-readable labels — handy for inference and debugging.

In [6]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_NAMES),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
model.to(DEVICE)

# The classification head is freshly initialized (warning about this is EXPECTED):
# we're about to fine-tune it. The base encoder weights are pretrained.
print(type(model).__name__, "loaded on", DEVICE)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 24056.81it/s]


[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification loaded on mps


In [7]:
# Inspect the model: size, head shape, and config labels.
n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {n_params:,}")
print(f"Trainable params: {n_trainable:,}")
print("Output labels   :", model.config.id2label)
print("Max input len   :", tokenizer.model_max_length)
print("\nClassifier head :", model.classifier)

Total params    : 66,955,779
Trainable params: 66,955,779
Output labels   : {0: 'negative', 1: 'neutral', 2: 'positive'}
Max input len   : 512

Classifier head : Linear(in_features=768, out_features=3, bias=True)


## 3. Wiring sanity check — tokenize + one forward pass

Run a single untrained forward pass to confirm tokenizer → model → logits → predicted
label all connect. Predictions will be random/garbage (the head is untrained) — that's
fine; we only care that shapes and the pipeline work.

In [8]:
examples = [
    "The company reported record quarterly profit and raised its dividend.",
    "Shares plunged after the firm warned of widening losses and layoffs.",
    "The board will meet on Thursday to review the quarterly filing.",
]

enc = tokenizer(examples, padding=True, truncation=True, max_length=128, return_tensors="pt")
print("input_ids shape:", enc["input_ids"].shape)  # (batch, seq_len)

model.eval()
with torch.no_grad():
    logits = model(**{k: v.to(DEVICE) for k, v in enc.items()}).logits

probs = torch.softmax(logits, dim=-1).cpu()
pred_ids = probs.argmax(dim=-1).tolist()

pd.DataFrame({
    "sentence": examples,
    "pred": [ID2LABEL[i] for i in pred_ids],
    **{f"p({name})": probs[:, i].round(decimals=3).tolist() for i, name in ID2LABEL.items()},
})

input_ids shape: torch.Size([3, 15])


,sentence,pred,p(negative),p(neutral),p(positive)
0,The company reported record quarterly profit a...,positive,0.271,0.346,0.383
1,Shares plunged after the firm warned of wideni...,positive,0.274,0.341,0.385
2,The board will meet on Thursday to review the ...,positive,0.272,0.349,0.379
